# 布尔 motif 因果图

这个 notebook 只保留两张主图、导出代码，以及和这两张图直接相关的数值输出。


In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import numpy as np

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'utils.py').exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break

from utils import (
    discrete_causal_graph,
    enumerate_binary_states,
    render_causal_graph_svg,
    render_ground_truth_causal_graph_svg,
)

try:
    from IPython.display import HTML, display
except Exception:  # pragma: no cover - notebook fallback
    HTML = None
    display = print

PROJECT_ROOT = Path(sys.path[0]).resolve()
FIG_DIR = (PROJECT_ROOT / 'fig' / 'boolean_motif_causal_graphs').resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')


def render_html_table(rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    if HTML is None:
        print(rows)
        return

    header = ''.join(f'<th style="padding:4px 8px">{key}</th>' for key in rows[0].keys())
    body = []
    for row in rows:
        cells = ''.join(
            f'<td style="padding:4px 8px">{value:.3f}</td>' if isinstance(value, float) else f'<td style="padding:4px 8px">{value}</td>'
            for value in row.values()
        )
        body.append(f'<tr>{cells}</tr>')
    html = '<table style="border-collapse:collapse" border="1"><tr>' + header + '</tr>' + ''.join(body) + '</table>'
    display(HTML(html))


def build_main_example() -> tuple[np.ndarray, list[tuple[int, int] | tuple[int, int, str]], list[dict[str, Any]]]:
    states = enumerate_binary_states(N_NODES)
    index_by_state = {tuple(state.tolist()): idx for idx, state in enumerate(states)}
    tpm = np.zeros((len(states), len(states)), dtype=float)

    for row_index, state in enumerate(states):
        x0, x1, x2 = (int(value) for value in state)
        next_state = (
            x1,
            int(x1 and x2),
            int((x0 + x1) % 2),
        )
        tpm[row_index, index_by_state[next_state]] = 1.0

    directed_edges: list[tuple[int, int] | tuple[int, int, str]] = [
        (1, 0, 'COPY'),
    ]
    hyperedges = [
        {'sources': (1, 2), 'target': 1, 'label': 'AND', 'value': 1.0},
        {'sources': (0, 1), 'target': 2, 'label': 'XOR', 'value': 1.0},
    ]
    return tpm, directed_edges, hyperedges


def select_single_target_hyperedges(summary: dict[str, Any]) -> list[dict[str, Any]]:
    label_by_target = {1: 'AND', 2: 'XOR'}
    rows = []
    for row in summary['hyperedges']:
        value = float(row['value'])
        if value <= HYPEREDGE_THRESHOLD:
            continue
        rows.append(
            {
                'sources': tuple(int(source) for source in row['sources']),
                'target': int(row['target']),
                'value': value,
                'label': label_by_target[int(row['target'])],
                'color': '',
            }
        )
    return rows


def select_multi_target_hyperedges(summary: dict[str, Any], max_target_dim: int | None = None) -> list[dict[str, Any]]:
    if max_target_dim is None:
        max_target_dim = N_NODES
    rows = []
    upper_target_order = min(N_NODES, max_target_dim)
    for target_order in range(2, upper_target_order + 1):
        for row in summary.get('hyperedges_by_target_order', {}).get(target_order, []):
            value = float(row['value'])
            if value <= HYPEREDGE_THRESHOLD:
                continue
            rows.append(
                {
                    'sources': tuple(int(source) for source in row['sources']),
                    'targets': tuple(int(target) for target in row['targets']),
                    'value': value,
                    'label': '',
                    'color': '',
                }
            )
    rows.sort(key=lambda item: (-item['value'], item['targets']))
    return rows


def format_pairwise_weight_rows(pairwise_matrix: np.ndarray) -> list[dict[str, Any]]:
    rows = []
    for target in range(pairwise_matrix.shape[1]):
        for source in range(pairwise_matrix.shape[0]):
            value = float(pairwise_matrix[source, target])
            if abs(value) <= EDGE_THRESHOLD:
                continue
            rows.append(
                {
                    '源节点': NODE_LABELS[source],
                    '目标节点': NODE_LABELS[target],
                    'Pairwise EI': value,
                }
            )
    rows.sort(key=lambda row: (-abs(row['Pairwise EI']), row['目标节点'], row['源节点']))
    return rows


def format_hyperedge_weight_rows(summary: dict[str, Any]) -> list[dict[str, Any]]:
    label_by_target = {1: 'AND', 2: 'XOR'}
    rows = []
    for row in summary['hyperedges']:
        value = float(row['value'])
        if abs(value) <= HYPEREDGE_THRESHOLD:
            continue
        rows.append(
            {
                '机制': label_by_target[int(row['target'])],
                '源节点组': ', '.join(NODE_LABELS[source] for source in row['sources']),
                '目标节点': NODE_LABELS[int(row['target'])],
                '协同权重': value,
            }
        )
    rows.sort(key=lambda row: (-abs(row['协同权重']), row['目标节点'], row['源节点组']))
    return rows


def format_multi_target_hyperedge_rows(summary: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    for target_order in (2, 3):
        for row in summary.get('hyperedges_by_target_order', {}).get(target_order, []):
            sources = tuple(int(source) for source in row['sources'])
            targets = tuple(int(target) for target in row['targets'])
            value = float(row['value'])
            if sources != (0, 1) or value <= HYPEREDGE_THRESHOLD:
                continue
            rows.append(
                {
                    '目标维度': len(targets),
                    '源节点组': ', '.join(NODE_LABELS[source] for source in sources),
                    '目标节点组': ', '.join(NODE_LABELS[target] for target in targets),
                    '联合EI': float(row['ei_joint']),
                    '单节点EI和': float(sum(row['single_eis'])),
                    '协同': value,
                }
            )
    rows.sort(key=lambda item: (item['目标维度'], item['目标节点组']))
    return rows


In [4]:
# 节点数量：当前主例子固定为 3 个变量。
N_NODES = 3
# 节点标签：对应图上的 x0, x1, x2。
NODE_LABELS = ['x0', 'x1', 'x2']
# 普通边显示阈值：低于该 EI 的单边不会画到图里。
EDGE_THRESHOLD = 0.08
# 协同超边显示阈值：低于该协同值的超边不会画到图里。
HYPEREDGE_THRESHOLD = 0.01
# 合并图里允许展示的超边最大目标维度：默认 1 表示只画单目标超边。
COMBINED_HYPEREDGE_MAX_TARGET_DIM = 1
# 图宽度：控制两张 SVG 图的整体横向尺寸。
FIGURE_WIDTH = 620
# 图高度：控制两张 SVG 图的整体纵向尺寸。
FIGURE_HEIGHT = 280
# 普通边基础线宽：所有蓝色普通边都会先从这个粗细起步。
PAIRWISE_WIDTH_BASE = 0.6
# 普通边线宽增益：EI 越大，在线宽上额外增加的幅度越大。
PAIRWISE_WIDTH_SCALE = 2.6
# 超边源侧线宽：控制虚线从源节点到中间汇合点时的粗细。
HYPEREDGE_STROKE_WIDTH = 1.8
# 超边目标侧线宽：控制虚线从中间汇合点指向目标节点时的粗细。
HYPEREDGE_TARGET_STROKE_WIDTH = 2.0
# 箭头头宽度：两张图里所有箭头共用，值越大箭头头越宽。
ARROW_MARKER_WIDTH = 12.0
# 箭头头高度：两张图里所有箭头共用，值越大箭头头越高。
ARROW_MARKER_HEIGHT = 9.0
# 箭头头锚点横坐标：控制箭头头在终点附近贴得多近。
ARROW_MARKER_REF_X = 7.0
# 箭头头锚点纵坐标：通常保持在箭头头高度的一半附近。
ARROW_MARKER_REF_Y = 3.0
# 普通边弯曲强度：值越大，不同行之间的蓝色弧线越弯。
PAIRWISE_CURVATURE_SCALE = 0.06
# 普通边起点偏移：控制蓝色箭头从左侧源节点出发时离圆心多远。
PAIRWISE_START_OFFSET = 16.0
# 普通边终点偏移：控制蓝色箭头在右侧目标节点前停止的位置。
PAIRWISE_END_OFFSET = 16.0
# 超边源侧偏移：控制虚线从左侧源节点伸出时离圆心多远。
HYPEREDGE_SOURCE_OFFSET = 14.0
# 超边目标侧偏移：控制虚线箭头在右侧目标节点前停止的位置。
HYPEREDGE_TARGET_OFFSET = 16.0
# 超边汇合点间隙：控制虚线进入/离开中间圆圈前留下的空隙。
HYPEREDGE_JUNCTION_GAP = 8.0
# 超边圆圈纵向间距：控制相邻超边圆圈中心在纵向上至少相隔多少。
HYPEREDGE_JUNCTION_VERTICAL_GAP = 22.0
# 超边汇合点横向位置：两张图共用，值越大中间圆圈越靠右。
ARROW_JUNCTION_X_FRAC = 0.36
# 超边汇合点纵向偏移：两张图共用，正值整体下移，负值整体上移。
ARROW_JUNCTION_Y_OFFSET = 15.0
# 合并图统一颜色：下面的合并图里普通边和超边共用这一种颜色。
COMBINED_EDGE_COLOR = '#547BB4'
# 合并图中的超边统一颜色：不再按 AND / XOR / 多目标类型区分颜色。
HYPEREDGE_COLOR = COMBINED_EDGE_COLOR

main_tpm, gt_edges, gt_hyperedges = build_main_example()
main_summary = discrete_causal_graph(main_tpm, n_nodes=N_NODES, target_orders=(1, 2, 3))
pairwise_weight_rows = format_pairwise_weight_rows(main_summary['pairwise_ei'])
hyperedge_weight_rows = format_hyperedge_weight_rows(main_summary)
multi_target_hyperedge_rows = format_multi_target_hyperedge_rows(main_summary)
combined_hyperedges = []
if COMBINED_HYPEREDGE_MAX_TARGET_DIM >= 1:
    combined_hyperedges.extend(select_single_target_hyperedges(main_summary))
combined_hyperedges.extend(
    select_multi_target_hyperedges(
        main_summary,
        max_target_dim=COMBINED_HYPEREDGE_MAX_TARGET_DIM,
    )
)

arrow_render_kwargs = {
    'width': FIGURE_WIDTH,
    'height': FIGURE_HEIGHT,
    'pairwise_width_base': PAIRWISE_WIDTH_BASE,
    'pairwise_width_scale': PAIRWISE_WIDTH_SCALE,
    'hyperedge_stroke_width': HYPEREDGE_STROKE_WIDTH,
    'hyperedge_target_stroke_width': HYPEREDGE_TARGET_STROKE_WIDTH,
    'arrow_marker_width': ARROW_MARKER_WIDTH,
    'arrow_marker_height': ARROW_MARKER_HEIGHT,
    'arrow_marker_ref_x': ARROW_MARKER_REF_X,
    'arrow_marker_ref_y': ARROW_MARKER_REF_Y,
    'pairwise_curvature_scale': PAIRWISE_CURVATURE_SCALE,
    'pairwise_start_offset': PAIRWISE_START_OFFSET,
    'pairwise_end_offset': PAIRWISE_END_OFFSET,
    'hyperedge_source_offset': HYPEREDGE_SOURCE_OFFSET,
    'hyperedge_target_offset': HYPEREDGE_TARGET_OFFSET,
    'hyperedge_junction_gap': HYPEREDGE_JUNCTION_GAP,
    'hyperedge_junction_vertical_gap': HYPEREDGE_JUNCTION_VERTICAL_GAP,
    'hyperedge_junction_x_frac': ARROW_JUNCTION_X_FRAC,
    'hyperedge_junction_y_offset': ARROW_JUNCTION_Y_OFFSET,
}

ground_truth_svg = render_ground_truth_causal_graph_svg(
    '',
    n_nodes=N_NODES,
    directed_edges=gt_edges,
    hyperedges=gt_hyperedges,
    node_labels=NODE_LABELS,
    subtitle='',
    **arrow_render_kwargs,
)
combined_ei_svg = render_causal_graph_svg(
    '',
    pairwise_matrix=main_summary['pairwise_ei'],
    hyperedges=combined_hyperedges,
    node_labels=NODE_LABELS,
    subtitle='',
    edge_threshold=EDGE_THRESHOLD,
    hyperedge_threshold=HYPEREDGE_THRESHOLD,
    show_edge_values=False,
    show_text_labels=False,
    pairwise_edge_color=COMBINED_EDGE_COLOR,
    default_hyperedge_color=COMBINED_EDGE_COLOR,
    **arrow_render_kwargs,
)

write_text(FIG_DIR / 'ground_truth_mechanism.svg', ground_truth_svg)
write_text(FIG_DIR / 'combined_ei_graph.svg', combined_ei_svg)

manifest = {
    'render_params': {
        'figure_width': FIGURE_WIDTH,
        'figure_height': FIGURE_HEIGHT,
        'pairwise_width_base': PAIRWISE_WIDTH_BASE,
        'pairwise_width_scale': PAIRWISE_WIDTH_SCALE,
        'hyperedge_stroke_width': HYPEREDGE_STROKE_WIDTH,
        'hyperedge_target_stroke_width': HYPEREDGE_TARGET_STROKE_WIDTH,
        'arrow_marker_width': ARROW_MARKER_WIDTH,
        'arrow_marker_height': ARROW_MARKER_HEIGHT,
        'arrow_marker_ref_x': ARROW_MARKER_REF_X,
        'arrow_marker_ref_y': ARROW_MARKER_REF_Y,
        'pairwise_curvature_scale': PAIRWISE_CURVATURE_SCALE,
        'pairwise_start_offset': PAIRWISE_START_OFFSET,
        'pairwise_end_offset': PAIRWISE_END_OFFSET,
        'hyperedge_source_offset': HYPEREDGE_SOURCE_OFFSET,
        'hyperedge_target_offset': HYPEREDGE_TARGET_OFFSET,
        'hyperedge_junction_gap': HYPEREDGE_JUNCTION_GAP,
        'hyperedge_junction_vertical_gap': HYPEREDGE_JUNCTION_VERTICAL_GAP,
        'arrow_junction_x_frac': ARROW_JUNCTION_X_FRAC,
        'arrow_junction_y_offset': ARROW_JUNCTION_Y_OFFSET,
    },
    'main_example': {
        'pairwise_threshold': EDGE_THRESHOLD,
        'combined_edge_color': COMBINED_EDGE_COLOR,
        'hyperedge_threshold': HYPEREDGE_THRESHOLD,
        'combined_hyperedge_max_target_dim': COMBINED_HYPEREDGE_MAX_TARGET_DIM,
        'hyperedge_color': HYPEREDGE_COLOR,
        'pairwise_weight_rows': pairwise_weight_rows,
        'hyperedge_weight_rows': hyperedge_weight_rows,
        'multi_target_hyperedge_rows': multi_target_hyperedge_rows,
    },
    'files': {
        'ground_truth': 'ground_truth_mechanism.svg',
        'combined_ei_graph': 'combined_ei_graph.svg',
    },
}
write_text(FIG_DIR / 'manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

if HTML is not None:
    blocks = []
    for svg in [ground_truth_svg, combined_ei_svg]:
        blocks.append("<div style='display:inline-block;vertical-align:top;margin:6px 10px 16px 0;'>" + svg + '</div>')
    display(HTML(''.join(blocks)))
else:
    print('已导出图像到', FIG_DIR)


In [5]:
if HTML is not None:
    display(HTML('<h4>Pairwise EI 边权重</h4>'))
else:
    print('Pairwise EI 边权重')
render_html_table(pairwise_weight_rows)

if HTML is not None:
    display(HTML('<h4>单目标协同超边</h4>'))
else:
    print('单目标协同超边')
render_html_table(hyperedge_weight_rows)

if HTML is not None:
    display(HTML('<h4>所有多目标协同超边</h4>'))
else:
    print('所有多目标协同超边')
render_html_table(multi_target_hyperedge_rows)


源节点,目标节点,Pairwise EI
x1,x0,1.000
x1,x1,0.311
x2,x1,0.311


机制,源节点组,目标节点,协同权重
XOR,"x0, x1",x2,1.000
AND,"x1, x2",x1,0.189


目标维度,源节点组,目标节点组,联合EI,单节点EI和,协同
2,"x0, x1","x1, x2",1.311,0.623,0.689
